In [ ]:
#!pip install sentence-transformers llama-index llama-index-llms-gemini  llama-index-embeddings-gemini llama-index-postprocessor-cohere-rerank

In [ ]:
# https://medium.com/@vamshidhar.pandrapagada/the-art-of-mining-hard-negatives-to-train-or-fine-tune-a-re-ranker-05db5259399e

# Finetuning reranker

While we won’t delve into the intricate design of each type of re-ranker, I believe it’s crucial for readers to grasp the variety of re-rankers available and understand the factors influencing their selection for different use cases.

- **Bi-Encoders**: They use separate encoders for query and document, generating independent embeddings for each. The similarity between these embeddings is then computed to determine relevance. These encoders are useful for semantic search but might not capture the full depth of query-document interactions.

- **Cross Encoders**: These models directly compute relevance scores by analyzing the query and document together. These scores effectively reorder the documents, placing the most relevant ones at the top. Their strength lies in capturing the nuanced interplay between query and document.

- **LLM-based Rerankers**: Large Language Models (LLMs) like GPT-3 can also be used for re-ranking. They offer the potential for greater contextual understanding and improved generalization but can be computationally demanding.




In the context of information retrieval and ranking :

- **Negative**: It does not belong to the positive class or is not the desired outcome. For instance, in a search scenario, a negative would be a document that’s not relevant to the user’s query.
- **Hard Negative**: It is very similar to a positive example, making it difficult for the reranker to correctly classify or rank it. This similarity can be in terms of content, context, or other features that the model uses for decision-making.

In [ ]:
import os

GOOGLE_API_KEY = "xxxxxxxxx"
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


COHERE_API_KEY = "xxxxxxxxxxxxxxx"
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [ ]:
from llama_index.core.evaluation import generate_question_context_pairs
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.postprocessor.cohere_rerank import CohereRerank
from sentence_transformers import CrossEncoder, InputExample
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core.node_parser import SentenceSplitter
from sentence_transformers import InputExample
from llama_index.llms.gemini import Gemini
from torch.utils.data import DataLoader
from llama_index.core import Settings
import pandas as pd
import random

## 1. Define the models and the cohere reranker

In [ ]:
# define the models
llm = Gemini(
    model="models/gemini-1.5-flash",
)

embed_model = GeminiEmbedding(
    model_name="models/embedding-001", api_key=GOOGLE_API_KEY
)
Settings.embed_model = embed_model
Settings.llm = llm


# define the cohere reranker
cohere_rerank = CohereRerank(api_key=COHERE_API_KEY, top_n=5 , model="rerank-v3.5") # top_n is the number of documents to rerank

## 2. Create the question-answer dataset

In [ ]:
def create_qa_dataset(input_dir ,  llm , num_questions_per_chunk):
    """
    Create a question-answer dataset from a directory of documents
    Args:
        input_dir: str
            The directory containing the documents
        llm: llama_index.llms.gemini.Gemini
            The LLM model to use for generating questions
        num_questions_per_chunk: int
            The number of questions to generate per chunk
    Returns:
        list: A list of questions
    """
    documents = SimpleDirectoryReader(input_dir=input_dir).load_data()
    node_parser = SentenceSplitter(chunk_size=256)
    nodes = node_parser.get_nodes_from_documents(documents)
    qa_dataset = generate_question_context_pairs(
          nodes, 
          llm=llm, 
          num_questions_per_chunk=num_questions_per_chunk
      )
    queries = qa_dataset.queries.values()
    return list(queries) , documents

## 3. Create the score query context dataset

In [ ]:
def create_score_query_context_dataset(queries , documents , split) : 
    """
    Create a dataset of queries, contexts and scores
    Args:
        queries: list
            A list of queries
        documents: list
            A list of documents
        split: str
            The split of the dataset
    Returns:
        pd.DataFrame: A DataFrame containing the queries, contexts and scores
    """
  contexts = []
  scores = []
  # create the index
  index = VectorStoreIndex.from_documents(documents)
  # create the retriever
  retriever = index.as_retriever(verbose=True, similarity_top_k=10)
  for query in queries:
      # retrieve the top 10 documents 
      nodes = retriever.retrieve(query)
      # rerank the documents using cohere and get the top 5 document
      response = cohere_rerank.postprocess_nodes(
                    nodes=nodes, query_str=query
                )
      random_number = random.randint(0, len(response)-1)
      contexts.append(response[random_number].text)
      scores.append(response[random_number].score)
  assert len(queries) == len(contexts) == len(scores)
  df = pd.DataFrame({"query": queries, "context": contexts, "score": scores})
  df.to_csv(f"{split}-data.csv", index=False)
  return df

- **Index the Documents**: The documents are indexed using a vector store, enabling efficient retrieval of relevant chunks.
- **Retrieve and Rerank**: For each query, the retriever fetches the top-k documents (e.g., top-10). These documents are then reranked using the Cohere reranker to refine the results.
- **Random Sampling**: To introduce diversity, a random document is selected from the reranked results. This ensures the model is exposed to a variety of query-context pairs.
- **Dataset Creation**: The queries, contexts, and their corresponding relevance scores are compiled into a DataFrame and saved as a CSV file for further use.


## 4. Create the train dataset

In [ ]:
# create the the train dataset
queries , documents = create_qa_dataset("/content/train_data", llm , 1 )
train_data = create_score_query_context_dataset(queries , documents , "train")
train_samples = [
    InputExample(texts=[row['query'], row['context']], label=row['score'])
    for _, row in train_data.iterrows()
]
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=8)

## 5. Create the validation dataset

In [ ]:
# create the validation dataset
queries , documents = create_qa_dataset("/content/val_data", llm , 1)
val_data = create_score_query_context_dataset(queries , documents , "validation")
val_samples = [
    InputExample(texts=[row['query'], row['context']], label=row['score'])
    for _, row in val_data.iterrows()
]
val_dataloader = DataLoader(val_samples, shuffle=True, batch_size=3)

## 6. Training & validation

In [ ]:
# Initialize a pre-trained Cross-Encoder model
model = CrossEncoder('BAAI/bge-reranker-base')

In [ ]:
from sentence_transformers.evaluation import SentenceEvaluator
import torch
from torch.utils.data import DataLoader
import logging
from sentence_transformers.util import batch_to_device
import os
import csv
from sentence_transformers import CrossEncoder
from tqdm.autonotebook import tqdm

logger = logging.getLogger(__name__)

class MSEEval(SentenceEvaluator):
    """
    Evaluate a model based on its accuracy on a labeled dataset

    This requires a model with LossFunction.SOFTMAX

    The results are written in a CSV. If a CSV already exists, then values are appended.
    """

    def __init__(self, 
                 dataloader: DataLoader, 
                 name: str = "", 
                 show_progress_bar: bool = True,
                 write_csv: bool = True):
        """
        Constructs an evaluator for the given dataset

        :param dataloader:
            the data for the evaluation
        """
        self.dataloader = dataloader
        self.name = name
        self.show_progress_bar = show_progress_bar

        if name:
            name = "_"+name

        self.write_csv = write_csv
        self.csv_file = "accuracy_evaluation"+name+"_results.csv"
        self.csv_headers = ["epoch", "steps", "accuracy"]

    def __call__(self, model: CrossEncoder, output_path: str = None, epoch: int = -1, steps: int = -1) -> float:
        model.model.eval()
        total = 0
        loss_total = 0

        if epoch != -1:
            if steps == -1:
                out_txt = " after epoch {}:".format(epoch)
            else:
                out_txt = " in epoch {} after {} steps:".format(epoch, steps)
        else:
            out_txt = ":"

        loss_fnc = torch.nn.MSELoss()
        activation_fnc = torch.nn.Sigmoid()

        logger.info("Evaluation on the "+self.name+" dataset"+out_txt)
        self.dataloader.collate_fn = model.smart_batching_collate
        for features, labels in tqdm(self.dataloader,  desc="Evaluation", smoothing=0.05, disable=not self.show_progress_bar):

            with torch.no_grad():
                model_predictions = model.model(**features, return_dict=True)
                logits = activation_fnc(model_predictions.logits)
                if model.config.num_labels == 1:
                    logits = logits.view(-1)
                loss_value = loss_fnc(logits, labels)

            total += 1 # number of batches
            loss_total += loss_value.cpu().item()
        mse = loss_total/total

        logger.info("MSE: {:.4f} ({}/{})\n".format(mse, loss_total, total))

        if output_path is not None and self.write_csv:
            csv_path = os.path.join(output_path, self.csv_file)
            if not os.path.isfile(csv_path):
                with open(csv_path, newline='', mode="w", encoding="utf-8") as f:
                    writer = csv.writer(f)
                    writer.writerow(self.csv_headers)
                    writer.writerow([epoch, steps, mse])
            else:
                with open(csv_path, newline='', mode="a", encoding="utf-8") as f:
                    writer = csv.writer(f)
                    writer.writerow([epoch, steps, mse])

        return mse

In [ ]:
# create the evaluator
evaluator = MSEEval(val_dataloader,show_progress_bar=True , write_csv=True )

In [ ]:
# Train the model
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=1,
    warmup_steps=100,
    evaluation_steps=3,
    output_path="my_model",
    save_best_model=True,
    use_amp=True,
    scheduler= 'warmupcosine',
    show_progress_bar=True,
)

- train_dataloader: The training data, processed in batches.
- evaluator: The custom evaluator to monitor performance during training.
- epochs: The number of training epochs (set to 1 in this case).
- warmup_steps: The number of warmup steps to stabilize training.
- evaluation_steps: The frequency of evaluation during training.
- output_path: The directory to save the fine-tuned model.
- save_best_model: Whether to save the best-performing model based on evaluation results.
- use_amp: Whether to use Automatic Mixed Precision (AMP) for faster training.
- scheduler: The learning rate scheduler (e.g., warmupcosine).
- show_progress_bar: Whether to display a progress bar during training.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

model.push_to_hub("user/bge-reranker-base-finetuned")

## 7. Use the finetune rerank model

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    model="user/bge-reranker-base-finetuned", top_n=3
)